In [ ]:

import json
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

def compile_predictions(predictions_file):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    results = []
    
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            # Get model generation output (usually first item)
            llm_output = data['model_generation'][0] if isinstance(data['model_generation'], list) else data['model_generation']
            answer = parse(llm_output, extraction_config=extraction_target)
            
            result = verify(gold, answer)
            results.append(result)
    
    accuracy = sum(results) / len(results) if results else 0
    return accuracy

from math import comb
def pass_at_k(n, c, k):
    """
    n: 总样本数
    c: 通过的样本数
    k: 评估的候选数
    """
    if n < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def compile_predictions_BoN_passK(predictions_file):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    results = []
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            o_results = []
            # Use all generations to verify
            for llm_output in data['model_generation']:
                answer = parse(llm_output, extraction_config=extraction_target)
                result = verify(gold, answer)
                o_results.append(result)
            results.append(o_results)
    
    # 计算BoN - 检查每个问题是否至少有一个正确
    BoN = sum(1 for o_result in results if any(o_result)) / len(results) if results else 0
    
    #  计算Pass@K - 统计所有通过的样本
    total_samples = sum(len(o_result) for o_result in results)
    correct_samples = sum(sum(o_result) for o_result in results)
    passK = pass_at_k(total_samples, correct_samples, 1)
    
    return BoN, passK


In [ ]:
import matplotlib.pyplot as plt

def draw_comparison_fig(baseline_x, baseline_y, seal_x, seal_y, x_label, y_label, title):
    plt.figure(figsize=(10, 5))
    plt.plot(baseline_x, baseline_y, marker='o', linestyle='-', color='b', label='Baseline')
    plt.plot(seal_x, seal_y, marker='o', linestyle='-', color='r', label='SEAL')
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.ylim(bottom=0, top=1)
    plt.legend()
    plt.show()

## Example

In [ ]:
path = "/media/volume/llm/llm_steering_reasoning/results/qwen3-baseline/Qwen/Qwen3-4B/gsm8k_test/1/0/8192/predictions.jsonl"
metric_path = "/media/volume/llm/llm_steering_reasoning/results/qwen3-baseline/Qwen/Qwen3-4B/gsm8k_test/1/0/8192/metrics.json"
accuracy = compile_predictions(path)
print("accuracy: ", accuracy)
# write to metric_path
with open(metric_path, 'w') as f:
    json.dump({"accuracy": accuracy}, f)
